<a href="https://colab.research.google.com/github/codehacker4655/codehacker4655/blob/main/CNN(PYTORCH).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets,transforms
from torchvision.utils import make_grid

In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
transfom=transforms.ToTensor()

In [4]:
train_data=datasets.MNIST(root='../cnn_data',train=True,download=True,transform=transfom)

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9.91M/9.91M [00:00<00:00, 16.2MB/s]


Extracting ../cnn_data/MNIST/raw/train-images-idx3-ubyte.gz to ../cnn_data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28.9k/28.9k [00:00<00:00, 490kB/s]


Extracting ../cnn_data/MNIST/raw/train-labels-idx1-ubyte.gz to ../cnn_data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1.65M/1.65M [00:00<00:00, 4.49MB/s]


Extracting ../cnn_data/MNIST/raw/t10k-images-idx3-ubyte.gz to ../cnn_data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4.54k/4.54k [00:00<00:00, 6.61MB/s]

Extracting ../cnn_data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ../cnn_data/MNIST/raw



In [5]:
test_data=datasets.MNIST(root='../cnn_data',train=False,download=True,transform=transfom)

In [6]:
train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: ../cnn_data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [7]:
test_data

Dataset MNIST
    Number of datapoints: 10000
    Root location: ../cnn_data
    Split: Test
    StandardTransform
Transform: ToTensor()

In [25]:
#create a small batch for images
batch_size=32
train_loader=DataLoader(train_data,batch_size=batch_size,shuffle=True)
test_loader=DataLoader(test_data,batch_size=batch_size,shuffle=False)

In [9]:
#definig that model
conv1=nn.Conv2d(1,6,3,1)#in_channel=(rgb or normal),out_channel(filters),(size of filter),stride,next is padding
conv2=nn.Conv2d(6,16,3,1)

In [10]:
for i,(X_train,y_train) in enumerate(train_loader):
    break

In [11]:
X_train.shape

torch.Size([1, 1, 28, 28])

In [12]:
x=X_train.view(1,1,28,28)

In [13]:
#perform our first convolution
x=F.relu(conv1(x))

In [14]:
#1 single image,6 is the filters we asked for,26*26 size
x.shape

torch.Size([1, 6, 26, 26])

In [15]:
#pass throught the pooling layer
x=F.max_pool2d(x,2,2)#kernel of 2 and stride of 2

In [16]:
x.shape #26/2 = 13

torch.Size([1, 6, 13, 13])

In [17]:
# do our second convolutional layer
x = F.relu(conv2(x))

In [18]:
x.shape#as we didn't set paddign we will loose 2 pixels

torch.Size([1, 16, 11, 11])

In [19]:
#pooling layer
x=F.max_pool2d(x,2,2)

In [20]:
x.shape # 11/2 =5.5

torch.Size([1, 16, 5, 5])

In [21]:
#modal class
class convolutionalNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1=nn.Conv2d(1,6,3,1)
    self.conv2=nn.Conv2d(6,16,3,1)
    #fully connected layer
    self.fc1=nn.Linear(5*5*16,120)
    self.fc2=nn.Linear(120,84)
    self.fc3=nn.Linear(84,10)

  def forward(self,x):
    x=F.relu(self.conv1(x))
    x=F.max_pool2d(x,2,2)
    #second pass
    x=F.relu(self.conv2(x))
    x=F.max_pool2d(x,2,2)
    #flatten
    x=x.view(-1,16*5*5)
    #fully connected layers
    x=F.relu(self.fc1(x))
    x=F.relu(self.fc2(x))
    x=self.fc3(x)
    return F.log_softmax(x,dim=1)

In [22]:
#create an instance of our model
torch.manual_seed(41)
model=convolutionalNetwork()
model

convolutionalNetwork(
  (conv1): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

In [23]:
#loss function optimizer
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.01)

In [26]:
import time
start_time=time.time()
#create variables to track things
epochs=5
train_losses=[]
test_losses=[]
train_correct=[]
test_correct=[]
#for loop of epochs
for i in range(epochs):
  trn_corr=0
  tst_corr=0


  #train
  for b,(X_train,y_train) in enumerate(train_loader):
    b+=1#start our batches at 1
    y_pred=model(X_train)
    loss=criterion(y_pred,y_train)
    predicted=torch.max(y_pred.data,1)[1]
    batch_corr=(predicted==y_train).sum()
    trn_corr+=batch_corr



    #update parameters
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if b%600==0:
      print(f'epoch:{i} batch:{b} loss:{loss.item()}')
  train_losses.append(loss)
  train_correct.append(trn_corr)




  #test
  with torch.no_grad():
    for b,(X_test,y_test) in enumerate(test_loader):
      y_val=model(X_test)
      predicted=torch.max(y_val.data,1)[1]
      tst_corr+=(predicted==y_test).sum()

loss=criterion(y_val,y_test)
test_losses.append(loss)
test_correct.append(tst_corr)












current_time=time.time()
total=current_time-start_time
print(f"training took: {total/60}minutes")

epoch:0 batch:600 loss:0.04500064253807068
epoch:0 batch:1200 loss:0.06381669640541077
epoch:0 batch:1800 loss:0.4226332902908325
epoch:1 batch:600 loss:0.1328839510679245
epoch:1 batch:1200 loss:0.17429743707180023
epoch:1 batch:1800 loss:0.052307259291410446
epoch:2 batch:600 loss:0.02383587881922722
epoch:2 batch:1200 loss:0.037008509039878845
epoch:2 batch:1800 loss:0.13890673220157623
epoch:3 batch:600 loss:0.2103486955165863
epoch:3 batch:1200 loss:0.055012959986925125
epoch:3 batch:1800 loss:0.15078827738761902
epoch:4 batch:600 loss:0.18135389685630798
epoch:4 batch:1200 loss:0.1838008016347885
epoch:4 batch:1800 loss:0.024890106171369553
training took: 2.0618154565493265minutes


In [29]:
#GRAB AN IMAGE
test_data[4143][0].reshape(28,28)


tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000

In [30]:
model.eval()
with torch.no_grad():
  output=model(test_data[4143][0].reshape(1,1,28,28))

In [31]:
output

tensor([[-16.2170, -11.7309, -15.5651, -18.9247,  -2.0609, -10.4713, -21.4382,
         -12.7908,  -7.3460,  -0.1370]])